# Notebook 01: Pemahaman Data & Exploratory Data Analysis (Bab 2 Laporan AM552)
**Mata Kuliah**: AM552 - Data Science of IS (Program Studi S1 Sistem Informasi, FTI UKSW)  
**Judul Proyek**: *Analisis Komparatif Sentimen Berbasis Aspek (ABSA) dan Pemodelan Topik Pengalaman Pengguna terhadap Ekosistem Aplikasi Mobile Generative AI di Indonesia*  
**Kode Kelompok**: `682024046_KEL_DS`

---
## 2.1 Sumber Data & 2.2 Kamus Data
Dataset diambil secara langsung (*100% real scraped data*) dari server **Google Play Store Indonesia (`lang='id'`, `country='id'`)** menggunakan pustaka `google-play-scraper` pada 8 aplikasi Generative AI terpopuler.

In [1]:
import pandas as pd
import numpy as np

RAW_CSV_PATH = '../dataset/RAW_DATASET_GENAI_INDONESIA_AM522B.csv'
df = pd.read_csv(RAW_CSV_PATH, low_memory=False)
print(f'Dimensi Dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom')
print(f'Unique reviewId : {df["reviewId"].nunique():,} (100% Unik)')
print(f'Rentang Tanggal : {df["at"].min()} s/d {df["at"].max()}')
df.head(5)[['reviewId', 'app_name', 'score', 'at', 'reviewCreatedVersion', 'content']]

Dimensi Dataset: 71,221 baris x 11 kolom
Unique reviewId : 71,221 (100% Unik)
Rentang Tanggal : 2023-05-31 07:37:14 s/d 2026-11-17 16:00:19

## 2.3 Kondisi Data (Missing Values, Duplikat, dan Ketimpangan Kelas)
Pemeriksaan menyeluruh terhadap kualitas data mentah sebelum masuk ke tahap persiapan data (Bab 3).

In [2]:
missing_df = pd.DataFrame({
    'Tipe Data': df.dtypes.astype(str),
    'Jumlah Missing': df.isna().sum(),
    'Persentase Missing (%)': (df.isna().sum() / len(df) * 100).round(2)
})
print('=== 1. AUDIT MISSING VALUES ===')
print(missing_df.to_string())

print('\n=== 2. DISTRIBUSI BARIS & RATING PER APLIKASI ===')
app_summary = df.groupby('app_name').agg(
    jumlah_baris=('reviewId', 'count'),
    rata_rata_rating=('score', 'mean')
).sort_values('jumlah_baris', ascending=False)
app_summary['proporsi_pct'] = (app_summary['jumlah_baris'] / len(df) * 100).round(2)
app_summary['rata_rata_rating'] = app_summary['rata_rata_rating'].round(2)
print(app_summary.to_string())

=== 1. AUDIT MISSING VALUES ===
                     Tipe Data  Jumlah Missing  Persentase Missing (%)
reviewId                   str               0                    0.00
app_name                   str               0                    0.00
app_id                     str               0                    0.00
userName                   str               1                    0.00
content                    str               0                    0.00
score                    int64               0                    0.00
thumbsUpCount            int64               0                    0.00
reviewCreatedVersion       str           13405                   18.82
at                         str               0                    0.00
replyContent               str           71053                   99.76
language_region            str               0                    0.00

=== 2. DISTRIBUSI BARIS & RATING PER APLIKASI ===
                           jumlah_baris  rata_rata_rating  propor

## 2.4 Eksplorasi Data & Interpretasi Grafik (2 Kalimat per Grafik Sesuai Template UKSW)
- **Grafik 1 (`01_distribusi_ulasan_per_aplikasi.png`)**: Grafik ini menunjukkan bahwa ChatGPT (`22.000` ulasan), Google Gemini (`12.600` ulasan), dan Character.AI (`12.000` ulasan) mendominasi `65,43%` dari total `71.221` ulasan aplikasi Generative AI di Indonesia. Hal ini berarti analisis harus memperhitungkan bobot volume per aplikasi serta kontras rating yang ekstrem antara asisten produktivitas (rata-rata rating `> 4.29`) dan Character.AI (`1.58`).
- **Grafik 2 (`02_distribusi_rating_per_aplikasi.png`)**: Grafik ini memperlihatkan ketimpangan kelas yang tajam di mana Bintang 5 mendominasi `63,96%` keseluruhan dataset, namun pada Character.AI justru Bintang 1 mencapai lebih dari `78%`. Implikasinya, evaluasi model klasifikasi pada Bab 4 dan Bab 5 tidak boleh hanya mengandalkan *Accuracy*, melainkan wajib menggunakan *Macro F1-Score*, *Balanced Accuracy*, dan *class weighting*.
- **Grafik 3 (`03_deteksi_awal_discrepancy_per_aplikasi.png`)**: Grafik ini mengungkap bahwa terdapat `3.271` ulasan (`4,59%`) yang mengalami *Rating-Text Discrepancy*, dengan proporsi tertinggi pada DeepSeek (`11,98%`) dan Claude (`9,05%`). Fenomena ini membuktikan bahwa banyak pengguna Indonesia tetap memberikan rating Bintang 4–5 meskipun isi ulasannya mengeluhkan masalah *server sibuk* atau *batasan kuota gratis*.